In [1]:

import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder


In [3]:
raw = pd.read_csv('food_recall.csv', dtype= {"event_id":"string"}).reset_index(names="raw_idx")

In [5]:
raw.head(3)

,raw_idx,status,city,state,country,classification,product_type,event_id,recalling_firm,address_1,...,recall_number,product_description,product_quantity,reason_for_recall,recall_initiation_date,center_classification_date,report_date,code_info,more_code_info,termination_date
0,0,Ongoing,San Francisco,CA,United States,Class III,Food,99859,"Kwok Shing, Inc.",1150 Thomas Ave,...,H-1393-2026,American Ginsengs. PRODUCT OF WISCONSIN. NET W...,70 boxes,Potential contamination with pesticides.,20260908,20260922.0,20260930,108-728-420,NaN,NaN
1,1,Ongoing,New York,NY,United States,Class I,Food,99853,GIAS FOOD INC.,360 Madison Ave Fl 7,...,H-1396-2026,Lemon Alfredo Fettuccine packed in a yellow pl...,"16,002 bags (2,286 cases)",Contamination with Listeria monocytogenes,20260903,20260924.0,20260930,L6080C SEP 20 2027 L6079C SEP 19 2027,NaN,NaN
2,2,Ongoing,Austin,TX,United States,Class I,Food,99874,"WFM Purchasing, LP",550 Bowie St,...,H-1395-2026,Fromagerie Coop de Mean Cabricharme and Belgiu...,21 Wheels,Undeclared Egg Allergen,20260908,20260922.0,20260930,"Shelf life: 30 days, all dates through and inc...",NaN,NaN


In [7]:
rmap = pd.read_csv('/Users/lehuuhung/assignment/ExpressGNN1/V_1/ExpressGNN/data/fda/record_id_mapping.csv')
rmap.head(3)

,record_id,source_csv_row,recall_number,event_id,event_internal_id
0,record_000001,2,H-1393-2026,99859,event_99859
1,record_000002,3,H-1396-2026,99853,event_99853
2,record_000003,4,H-1395-2026,99874,event_99874


In [9]:
rmap["raw_idx"] = rmap['source_csv_row'] - 2
raw = raw.merge(rmap[["record_id", "event_id", "raw_idx"]], on="raw_idx", how="left", suffixes=("", "_m"))
raw.head(3)

,raw_idx,status,city,state,country,classification,product_type,event_id,recalling_firm,address_1,...,product_quantity,reason_for_recall,recall_initiation_date,center_classification_date,report_date,code_info,more_code_info,termination_date,record_id,event_id_m
0,0,Ongoing,San Francisco,CA,United States,Class III,Food,99859,"Kwok Shing, Inc.",1150 Thomas Ave,...,70 boxes,Potential contamination with pesticides.,20260908,20260922.0,20260930,108-728-420,NaN,NaN,record_000001,99859
1,1,Ongoing,New York,NY,United States,Class I,Food,99853,GIAS FOOD INC.,360 Madison Ave Fl 7,...,"16,002 bags (2,286 cases)",Contamination with Listeria monocytogenes,20260903,20260924.0,20260930,L6080C SEP 20 2027 L6079C SEP 19 2027,NaN,NaN,record_000002,99853
2,2,Ongoing,Austin,TX,United States,Class I,Food,99874,"WFM Purchasing, LP",550 Bowie St,...,21 Wheels,Undeclared Egg Allergen,20260908,20260922.0,20260930,"Shelf life: 30 days, all dates through and inc...",NaN,NaN,record_000003,99874


In [10]:
raw['event_id'] = raw["event_id_m"].astype("string")

In [ ]:
rc, cc, fm, ev ={}, {}, {}, {}

mystring = Path(/Users/lehuuhung/assignment/ExpressGNN1/V_1/ExpressGNN/data/fda/parser_test/facts.txt)

for line in mystring.read_text().splitlines():
    m = re.fullmatch(r"(\w+)\(([^,]+),\s*([^\)]+)\)", line.strip())
    if not m:
        continue
    pred, arg1, arg2 = m.groups(1), m.groups(2), m.groups(3)
    if pred == "has_reason_category":
        rc[arg1] = arg2
    elif pred == "has_classification":
        cc[arg1] = arg2
    elif pred == "recalled_by":
        fm[arg1] = arg2
    elif pred == "belongs_to_event":
        ev[arg1] = arg2
raw["rc"] = raw["record_id"].map(rc)
raw["cc"] = raw["record_id"].map(cc)
raw["fm"] = raw["record_id"].map(fm)
raw["ev"] = raw["record_id"].map(ev)

valid = (
        raw[raw["cc"].isin(["Class I", "Class II", "Class III"] & raw["rc"].notna())]
        .copy()
        .reset_index(drop=True)
)

FileNotFoundError: [Errno 2] No such file or directory: 'fda/parser_test/facts.txt'

In [16]:
mystring= '/Users/lehuuhung/assignment/ExpressGNN1/V_1/ExpressGNN/data/fda/parser_test/facts.txt'

mystring.read_text().splitlines()

AttributeError: 'str' object has no attribute 'read_text'